### Steps for Converting Zip/Tract Codes for 2020-2022 Data:

#### Converting to Census Codes

Notes:
- Use the US Census Bureau API geocoder instead of Geocod.io
- This site only allows geocoding for 10,000 records at a time
- Must split the csv if we want this to work

1) Read the data
2) Check the data and see if imputing is necessary
3) Split the data
4) Access the API, define a function, and geocode each split
5) Concatenate the splits

#### Converting to Zip Codes

Notes:
- For this step, we will use geopandas and shapely.geometry to convert
- It's not as complex as census codes; we can submit all the records at once
- Just make sure you use the right shape file

### PART 1: Census Tract Codes

In [30]:
import pandas as pd
import numpy as np

In [3]:
# Reading the data
data_2020 = pd.read_csv('Chicago_Data_2020 - Chicago_Data_2020.csv')
data_2021 = pd.read_csv('Chicago_Data_2021 - Chicago_asset_data_2021.csv.csv')
data_2022 = pd.read_csv('Chicago_Data_2022 - Chicago_Data_2022.csv.csv')

In [4]:
# Quick analysis

# 2020 data
display(data_2020.dtypes)
display(data_2020.isnull().sum())
# all of the numerical values are floats or ints
# there are 39 missing lat and lang values

Name                 object
BuildingNumber        int64
Street               object
Unit                 object
Fraction             object
GeoArea              object
City                 object
Zip                   int64
PlaceStatus          object
PlaceType            object
PlaceSubType         object
PrivateResidence     object
MappedYear            int64
Note                 object
PhoneNumber          object
Email                object
Geolocation          object
Lng                 float64
Lat                 float64
Website              object
dtype: object

Name                    0
BuildingNumber          0
Street                  0
Unit                 9377
Fraction            10163
GeoArea                 0
City                    0
Zip                     0
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                 8297
PhoneNumber           644
Email               10050
Geolocation            39
Lng                    39
Lat                    39
Website              8263
dtype: int64

In [5]:
# 2021 data
display(data_2021.dtypes)
display(data_2021.isnull().sum())
# all of the numerical values are floats or ints
# there are 215 missing lat and lang values

Name                  object
Building Number        int64
Street                object
Unit                  object
Fraction              object
Geo Area              object
City                  object
State                 object
Suggested Address    float64
Phone Number          object
Place Type            object
Place Subtype         object
Place Status          object
Review Status         object
Private Residence     object
Research Type        float64
Note                  object
Zip                  float64
Email                 object
Geo Location          object
Lng                  float64
Lat                  float64
Website               object
Mapped Date           object
Updated Date          object
IsNormalized            bool
dtype: object

Name                     0
Building Number          0
Street                   0
Unit                 20477
Fraction             23090
Geo Area                 0
City                     0
State                    0
Suggested Address    23122
Phone Number          1809
Place Type               2
Place Subtype            6
Place Status             0
Review Status            0
Private Residence        0
Research Type        23122
Note                 13810
Zip                    167
Email                22132
Geo Location           215
Lng                    215
Lat                    215
Website              16972
Mapped Date              0
Updated Date             0
IsNormalized             0
dtype: int64

In [6]:
# 2022 data
display(data_2022.dtypes)
display(data_2022.isnull().sum())
# all of the numerical values are floats or ints
# there are 3580 missing lat and lang values

Name                  object
Building Number       object
Street                object
Unit                  object
Fraction              object
Geo Area              object
City                  object
State                 object
Phone Number          object
Place Type            object
Place Subtype         object
Place Status          object
Review Status         object
Private Residence     object
Note                  object
Zip                  float64
Email                 object
Geo Location          object
Lng                  float64
Lat                  float64
Website               object
Mapped Date           object
Updated Date          object
Is New Place         float64
IsNormalized          object
dtype: object

Name                    15
Building Number          0
Street                   0
Unit                 21404
Fraction             22688
Geo Area               124
City                    76
State                   76
Phone Number          4167
Place Type              22
Place Subtype          159
Place Status             9
Review Status            0
Private Residence     2279
Note                 18316
Zip                    954
Email                22220
Geo Location          3584
Lng                   3580
Lat                   3580
Website              18522
Mapped Date           2799
Updated Date          2799
Is New Place         19688
IsNormalized          2799
dtype: int64

In [7]:
# I can't manually impute lat and lang values for over 4000 building numbers and street addresses
# Let's see if we can do this another way

# get only nulls
null_2020_data = data_2020[data_2020['Lng'].isnull()]

In [8]:
!pip install geopy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from geopy.geocoders import Nominatim
import time

null_2020_data = null_2020_data.copy()

geolocator = Nominatim(user_agent="geo")

def geocode(row):
    try:
        address = f"{row['Street']} {row['BuildingNumber']}, {row['City']}"
        location = geolocator.geocode(address, timeout=10)
        time.sleep(1)   # respect API rate limit
        
        if location:
            return pd.Series([location.longitude, location.latitude])
    except:
        pass
        
    return pd.Series([None, None])

null_2020_data[['Lng','Lat']] = null_2020_data.apply(geocode, axis=1)
null_2020_data

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
10134,Catholic Charities Archdiocese of Chicago,5645,W. Corcoran,NaN,NaN,Austin,Chicago,60644,Active,Social Services & Political Advocacy,Social service,No,2020,NaN,773-922-7200,NaN,NaN,-87.772737,41.887105,NaN
10135,DTA Development,1059,N Laramie Ave,NaN,NaN,Austin,Chicago,60644,Active,Trade Service,Building trades,No,2020,NaN,773-287-4710,NaN,NaN,-87.755430,41.900295,NaN
10136,Lawndale Heritage Garden,1509,S Lawndale Ave,NaN,NaN,North Lawndale,Chicago,60623,Active,Other,Community garden,No,2020,NaN,NaN,NaN,NaN,-87.717580,41.860565,NaN
10137,Lawndale Community Academy,3500,W Douglas Blvd,NaN,NaN,North Lawndale,Chicago,60623,Active,Childcare and Schools,Public school K-12,No,2020,NaN,773-534-1635,NaN,NaN,-87.712824,41.863269,lawndalepanthers.org
10138,Indigo Remodeling,2532,W Lyndale Ave,NaN,NaN,Logan Square,Chicago,60647,Active,Trade Service,Building trades,Yes,2020,NaN,773-489-3830,NaN,NaN,-87.740121,41.921576,indigoremodeling.com
10139,Chicago municipal employees credit union,6612,W north ave,NaN,NaN,Austin,Chicago,60651,Active,"Financial, Insurance, Real Estate, Legal, and ...",Bank,No,2020,NaN,312-236-2326,NaN,NaN,-87.790389,41.909166,NaN
10140,H and R Block,143,W Roosevelt Rd,NaN,NaN,Austin,Chicago,60644,Active,"Financial, Insurance, Real Estate, Legal, and ...",Accountant,No,2020,NaN,630-231-7445,NaN,NaN,-87.631991,41.867505,NaN
10141,Express Tax,5618,W Chicago Ave,NaN,NaN,Austin,Chicago,60644,Active,"Financial, Insurance, Real Estate, Legal, and ...",Accountant,No,2020,NaN,773-564-3163,NaN,NaN,-87.766215,41.894924,NaN
10142,DMC,2222,N Elston Ave,NaN,NaN,Logan Square,Chicago,60647,Active,Trade Service,Tech service,No,2020,NaN,312-872-0069,sales.inquiry@dmcinfo.com,NaN,-87.673182,41.921729,https://www.dmcinfo.com/
10143,Nado Healthcare,1550,S Albany Ave,NaN,NaN,North Lawndale,Chicago,60623,Active,Service or Programmed Residential Space,Group home,Yes,2020,NaN,773-277-6868,NaN,NaN,-87.703262,41.859874,madohealthcare.com


In [11]:
notnull_2020_data = data_2020[data_2020['Lng'].notnull()]
notnull_2020_data

,Name,BuildingNumber,Street,Unit,Fraction,GeoArea,City,Zip,PlaceStatus,PlaceType,PlaceSubType,PrivateResidence,MappedYear,Note,PhoneNumber,Email,Geolocation,Lng,Lat,Website
0,Moped,2068,N Western Ave,Ste 1,NaN,Logan Square,Chicago,60647,Active,Trade Service,Bike shops,No,2020,TEMP CLOSED,773-252-6705,NaN,POINT(-87.6879466 41.9196014),-87.687947,41.919601,enjoymoped.com
1,Grime A Way Professional Washing,4521,N Greenview Ave,NaN,NaN,Uptown,Chicago,60640,Out of Business,Personal Service,"Laundry, tailor",No,2020,NaN,773-297-4950,NaN,POINT(-87.6677255605062 41.9640845870442),-87.667726,41.964085,NaN
2,Bff Bikes,2113,W Armitage Ave,NaN,NaN,Logan Square,Chicago,60647,Active,Trade Service,Bike shops,No,2020,NaN,773-666-5153,NaN,POINT(-87.6806788 41.9175236),-87.680679,41.917524,bffbikes.com
3,John Culver Attorney,4707,N Broadway,NaN,NaN,Uptown,Chicago,60640,Active,"Financial, Insurance, Real Estate, Legal, and ...",Lawyer's office,No,2020,NaN,773-370-0273,NaN,POINT(-87.658832275126 41.9675943034016),-87.658832,41.967594,NaN
4,Archangel Trust,5000,N Marine Dr,NaN,NaN,Uptown,Chicago,60640,Unknown,"Financial, Insurance, Real Estate, Legal, and ...",Other legal,Yes,2020,"No website, no social media, and called 312-88...",312-883-5280,NaN,POINT(-87.6510547404478 41.9734008529356),-87.651055,41.973401,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10129,Lawndale Christian Learning Center,3841,W Ogden Ave,NaN,NaN,North Lawndale,Chicago,60623,Active,Social Services & Political Advocacy,Youth Centers,No,2020,NaN,NaN,NaN,POINT(-87.7213424 41.8519455),-87.721342,41.851945,NaN
10130,GROWcommunity,4550,N Hermitage Ave,NaN,NaN,Uptown,Chicago,60640,Out of Business,Social Services & Political Advocacy,Youth Centers,No,2020,"No online presence, deactivated number, new ch...",773-561-8874,NaN,POINT(-87.6725199401444 41.9649384970139),-87.672520,41.964939,NaN
10131,Life Changing Community Outreach,5902,W North Ave,NaN,NaN,Austin,Chicago,60639,Active,Social Services & Political Advocacy,Youth Centers,No,2020,NaN,312-468-4723,NaN,POINT(-87.7729400257946 41.9092964716304),-87.772940,41.909296,NaN
10132,Mathnasium,1754,W Wilson Ave,NaN,NaN,Uptown,Chicago,60640,Active,Social Services & Political Advocacy,Youth Centers,No,2020,Temp Closed. Online Services.,773-880-6284,NaN,POINT(-87.672941532982 41.9652628280852),-87.672942,41.965263,mathnasium.com


In [37]:
cleaned_2020 = pd.DataFrame(pd.concat([notnull_2020_data,null_2020_data]))

In [23]:
# do the same for 2021 and 20222

import time
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geo")

def geocode(row):
    try:
        address = f"{row['Street']} {row['Building Number']}, {row['City']}, IL, USA"
        location = geolocator.geocode(address, timeout=10)
        time.sleep(1)

        if location:
            return pd.Series([location.longitude, location.latitude])

    except Exception:
        pass

    return pd.Series([None, None])


null_2021_data = data_2021[data_2021['Lng'].isnull()].copy()
null_2021_data[['Lng','Lat']] = null_2021_data.apply(geocode, axis=1)

null_2022_data = data_2022[data_2022['Lng'].isnull()].copy()
null_2022_data[['Lng','Lat']] = null_2022_data.apply(geocode, axis=1)

print(null_2021_data.shape)
print(null_2022_data.shape)

(215, 26)
(3580, 25)


In [24]:
null_2021_data

,Name,Building Number,Street,Unit,Fraction,Geo Area,City,State,Suggested Address,Phone Number,...,Note,Zip,Email,Geo Location,Lng,Lat,Website,Mapped Date,Updated Date,IsNormalized
8489,Lawndale Heritage Garden,1509,S Lawndale Ave,NaN,NaN,North Lawndale,Chicago,IL,NaN,NaN,...,NaN,60623.0,NaN,NaN,-87.717580,41.860565,https://abstractdecors.wordpress.com/sites-2/s...,7/8/2021 15:47,7/8/2021 15:47,False
8494,Express Tax,5618,W Chicago Ave,NaN,NaN,Austin,Chicago,IL,NaN,773-564-3163,...,NaN,60644.0,NaN,NaN,-87.766215,41.894924,NaN,6/15/2021 20:04,6/15/2021 20:04,False
8496,Nado Healthcare,1550,S Albany Ave,NaN,NaN,North Lawndale,Chicago,IL,NaN,773-277-6868,...,NaN,60623.0,NaN,NaN,-87.703262,41.859874,madohealthcare.com,7/1/2021 21:16,7/1/2021 21:16,False
8499,Hyde Park - Kenwood Community Conference,1525,E 53rd Street,907,NaN,Hyde Park,Chicago,IL,NaN,773-288-8343,...,NaN,60615.0,info@hydepark.org,NaN,-87.588195,41.799345,hydepark.org,7/13/2021 15:18,7/13/2021 15:18,False
8500,Neighborhood Housing Services,3601,W Chicago Ave,NaN,NaN,Austin,Chicago,IL,NaN,773-329-4111,...,NaN,60644.0,NaN,NaN,-87.716724,41.895103,NaN,6/16/2021 13:58,6/16/2021 13:58,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23117,Real Estate to Freedom,6906,S Bennett Ave,NaN,NaN,South Shore,Chicago,IL,NaN,855-595-3542,...,NaN,60649.0,NaN,NaN,-87.579159,41.769490,https://realestate2freedom.com,8/20/2021 13:54,1/1/1900 0:00,False
23118,Yephias Complete Studio,1873,E 71st St,NaN,NaN,South Shore,Chicago,IL,NaN,773-510-9355,...,NaN,60649.0,NaN,NaN,-87.578709,41.765965,https://styledbymsketa773-510-9355.as.me/sched...,8/20/2021 14:09,1/1/1900 0:00,False
23119,Mr. James Hair Studio,2053,E 75th St,NaN,NaN,South Shore,Chicago,IL,NaN,773-637-7750,...,NaN,60649.0,NaN,NaN,-87.573922,41.758713,NaN,8/20/2021 15:23,1/1/1900 0:00,False
23120,Greater Emmanuel (GEM) M.B.C.,252,E 115th St,NaN,NaN,Roseland,Chicago,IL,NaN,872-228-1332,...,NaN,60628.0,NaN,NaN,-87.616353,41.685472,http://greatere252.org,8/23/2021 15:03,1/1/1900 0:00,False


In [35]:
notnull_2021_data = data_2021[data_2021['Lng'].notnull()]
cleaned_2021 = pd.DataFrame(pd.concat([notnull_2021_data,null_2021_data]))

notnull_2022_data = data_2022[data_2022['Lng'].notnull()]
cleaned_2022 = pd.DataFrame(pd.concat([notnull_2022_data,null_2022_data]))

In [38]:
cleaned_2020.to_csv("cleaned_2020.csv", index=False)
cleaned_2021.to_csv("cleaned_2021.csv", index=False)
cleaned_2022.to_csv("cleaned_2022.csv", index=False)

#save as csv just in case

In [51]:
display(cleaned_2020.isnull().sum())
display(cleaned_2021.isnull().sum())
display(cleaned_2022.isnull().sum())

Name                    0
BuildingNumber          0
Street                  0
Unit                 9377
Fraction            10163
GeoArea                 0
City                    0
Zip                     0
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                 8297
PhoneNumber           644
Email               10050
Geolocation            39
Lng                     1
Lat                     1
Website              8263
dtype: int64

Name                     0
Building Number          0
Street                   0
Unit                 20477
Fraction             23090
Geo Area                 0
City                     0
State                    0
Suggested Address    23122
Phone Number          1809
Place Type               2
Place Subtype            6
Place Status             0
Review Status            0
Private Residence        0
Research Type        23122
Note                 13810
Zip                    167
Email                22132
Geo Location           215
Lng                      7
Lat                      7
Website              16972
Mapped Date              0
Updated Date             0
IsNormalized             0
dtype: int64

Name                    15
Building Number          0
Street                   0
Unit                 21404
Fraction             22688
Geo Area               124
City                    76
State                   76
Phone Number          4167
Place Type              22
Place Subtype          159
Place Status             9
Review Status            0
Private Residence     2279
Note                 18316
Zip                    954
Email                22220
Geo Location          3584
Lng                     92
Lat                     92
Website              18522
Mapped Date           2799
Updated Date          2799
Is New Place         19688
IsNormalized          2799
dtype: int64

In [52]:
# most of the nulls have now been imputed. There is still a couple nulls that the geocoding may have missed but can ask Noah about that 

In [57]:
# Splitting the data for 10,000 records at a time in the census geocoder
print(cleaned_2020.shape)
print(cleaned_2021.shape)
print(cleaned_2022.shape)

# will have to split 2020 in 2, and 2021 and 2022 in 3

(10173, 20)
(23122, 26)
(22786, 25)


In [64]:
# 2020

parts = np.array_split(cleaned_2020, 2)

# Save each part
for i, part in enumerate(parts):
    part.to_csv(f'split_file_2020_{i+1}.csv', index=False)

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [63]:
# 2021

parts = np.array_split(cleaned_2021, 3)

# Save each part
for i, part in enumerate(parts):
    part.to_csv(f'split_file_2021_{i+1}.csv', index=False)

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [62]:
# 2022

parts = np.array_split(cleaned_2022, 3)

# Save each part
for i, part in enumerate(parts):
    part.to_csv(f'split_file_2022_{i+1}.csv', index=False)

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [67]:
import requests
url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates" # url for the geocoder
def get_tract(Lat, Lng):
    params = {
        "x": Lng,
        "y": Lat,
        "benchmark": "Public_AR_Current",
        "vintage": "Census2020_Current",
        "format": "json"
    }
    r = requests.get(url, params=params)
    try:
        return r.json()["result"]["geographies"]["Census Tracts"][0]["GEOID"]
    except (KeyError, IndexError):
        return None

In [69]:
split_2020_1 = pd.read_csv('split_file_2020_1.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2020_1[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2020_1["tract_geoid"] = split_2020_1.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [72]:
split_2020_2 = pd.read_csv('split_file_2020_2.csv')

unique_coords = split_2020_2[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

def fetch_tract(coord):
    lat, lng = coord["Lat"], coord["Lng"]
    if pd.isna(lat) or pd.isna(lng):
        return (lat, lng), None
    return (lat, lng), get_tract(lat, lng)

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(fetch_tract, coord_list)

for k, v in results:
    coord_to_tract[k] = v

split_2020_2["tract_geoid"] = split_2020_2.apply(
    lambda r: coord_to_tract.get((r["Lat"], r["Lng"]), None),
    axis=1
)

In [75]:
data_2020_NEW = pd.concat([split_2020_1,split_2020_2])

In [76]:
# 2021 Data:
split_2021_1 = pd.read_csv('split_file_2021_1.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2021_1[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2021_1["tract_geoid"] = split_2021_1.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [77]:
split_2021_2 = pd.read_csv('split_file_2021_2.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2021_2[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2021_2["tract_geoid"] = split_2021_2.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [79]:
split_2021_3 = pd.read_csv('split_file_2021_3.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2021_3[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2021_3["tract_geoid"] = split_2021_3.apply(
    lambda r: coord_to_tract.get((r["Lat"], r["Lng"]), None),
    axis=1
)

In [81]:
data_2021_NEW = pd.concat([split_2021_1,split_2021_2,split_2021_3])

In [83]:
data_2021_NEW.shape

(23122, 27)

In [84]:
data_2021.shape

(23122, 26)

In [85]:
# 2022 data

split_2022_1 = pd.read_csv('split_file_2022_1.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2022_1[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2022_1["tract_geoid"] = split_2022_1.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)


In [86]:
split_2022_2 = pd.read_csv('split_file_2022_2.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2022_2[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2022_2["tract_geoid"] = split_2022_2.apply(
    lambda r: coord_to_tract[(r["Lat"], r["Lng"])],
    axis=1
)

In [87]:
split_2022_3 = pd.read_csv('split_file_2022_3.csv')

from concurrent.futures import ThreadPoolExecutor

unique_coords = split_2022_3[["Lat", "Lng"]].drop_duplicates()
coord_list = unique_coords.to_dict("records")

coord_to_tract = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    results = executor.map(
        lambda r: ((r["Lat"], r["Lng"]), get_tract(r["Lat"], r["Lng"])),
        coord_list
    )

for k, v in results:
    coord_to_tract[k] = v

split_2022_3["tract_geoid"] = split_2022_3.apply(
    lambda r: coord_to_tract.get((r["Lat"], r["Lng"]), None),
    axis=1
)

In [88]:
data_2022_NEW = pd.concat([split_2022_1,split_2022_2,split_2022_3])

In [89]:
data_2022_NEW.shape

(22786, 26)

In [90]:
data_2022.shape

(22786, 25)

In [ ]:
# now we have converted all 3 files to tract codes

#### Converting to Zip Codes

- Do the same but for zip codes now...some values already have zip included but must now impute the nulls
- We're going to do this using the 2020 ZCTA TIGER/Line Shapefiles: https://www.census.gov/programs-surveys/geography/guidance/geo-areas/zctas.html
- Also using the geopandas package

In [91]:
data_2020_NEW['Zip'].unique()

array([60647, 60640, 60625, 60630, 60614, 60644, 60651, 60624, 60632,
       60636, 60639, 60707, 60618, 60622, 60620, 60623, 60653, 60621,
       60615, 60637, 60617, 60649, 60608, 60612, 60827, 60609, 60613,
       60641, 60642, 60657, 60628, 60619, 60643, 60610, 60603, 60659,
       60601, 60654, 60606, 60604, 60646, 60626, 60611, 60616, 60602,
       60661, 60605, 60660])

In [92]:
data_2020_NEW.isnull().sum()

Name                    0
BuildingNumber          0
Street                  0
Unit                 9377
Fraction            10163
GeoArea                 0
City                    0
Zip                     0
PlaceStatus             0
PlaceType               0
PlaceSubType            0
PrivateResidence        0
MappedYear              0
Note                 8297
PhoneNumber           644
Email               10050
Geolocation            39
Lng                     1
Lat                     1
Website              8263
tract_geoid             1
dtype: int64

In [93]:
# data_2020 has accurate zip codes and no nulls for the zip

In [94]:
data_2021_NEW['Zip'].unique()

array([60647., 60625., 60630., 60614., 60644., 60651., 60624., 60632.,
       60636., 60639., 60707., 60618., 60622., 60620., 60623., 60653.,
       60621., 60615., 60637., 60617., 60649., 60608., 60612., 60609.,
       60641., 60642., 60619., 60643., 60640., 60603., 60659., 60657.,
       60601., 60654., 60606., 60604., 60646., 60611., 60602., 60605.,
       60616., 60661., 60628., 60607., 60633., 60629., 60652.,    nan,
       60638., 60631., 60827., 60613., 60185., 60655., 60016., 60656.,
       60501., 60482., 60458.])

In [96]:
data_2021_NEW.isnull().sum()

Name                     0
Building Number          0
Street                   0
Unit                 20477
Fraction             23090
Geo Area                 0
City                     0
State                    0
Suggested Address    23122
Phone Number          1809
Place Type               2
Place Subtype            6
Place Status             0
Review Status            0
Private Residence        0
Research Type        23122
Note                 13810
Zip                    167
Email                22132
Geo Location           215
Lng                      7
Lat                      7
Website              16972
Mapped Date              0
Updated Date             0
IsNormalized             0
tract_geoid              7
dtype: int64

In [98]:
#data 2021 has 167 nulls for the zip codes
# will need to impute and convert floats to ints
data_2021_NEW['Zip'].dtypes

dtype('float64')

In [103]:
# we are going to make a new col for zip codes:

import geopandas as gpd
from shapely.geometry import Point

gdf_points = gpd.GeoDataFrame(
    data_2021_NEW,
    geometry=gpd.points_from_xy(data_2021_NEW["Lng"], data_2021_NEW["Lat"]),
    crs="EPSG:4326"  # WGS84 (standard lat/lon)
)

zcta = gpd.read_file('tl_2020_us_zcta520.shp')

zcta.columns

zcta = zcta.to_crs(gdf_points.crs)

gdf_with_zip = gpd.sjoin(
    gdf_points,
    zcta[["ZCTA5CE20", "geometry"]],
    how="left",
    predicate="within"
)

data_2021_FINAL = gdf_with_zip.drop(
    columns=["geometry", "index_right"]
)

In [105]:
data_2021_NEW.isnull().sum()
# tract_geoid has 7 nulls but that means we imputed most of the zips

Name                     0
Building Number          0
Street                   0
Unit                 20477
Fraction             23090
Geo Area                 0
City                     0
State                    0
Suggested Address    23122
Phone Number          1809
Place Type               2
Place Subtype            6
Place Status             0
Review Status            0
Private Residence        0
Research Type        23122
Note                 13810
Zip                    167
Email                22132
Geo Location           215
Lng                      7
Lat                      7
Website              16972
Mapped Date              0
Updated Date             0
IsNormalized             0
tract_geoid              7
dtype: int64

In [106]:
# same for 2022 data

import geopandas as gpd
from shapely.geometry import Point

gdf_points = gpd.GeoDataFrame(
    data_2022_NEW,
    geometry=gpd.points_from_xy(data_2022_NEW["Lng"], data_2022_NEW["Lat"]),
    crs="EPSG:4326"  # WGS84 (standard lat/lon)
)

zcta = gpd.read_file('tl_2020_us_zcta520.shp')

zcta.columns

zcta = zcta.to_crs(gdf_points.crs)

gdf_with_zip = gpd.sjoin(
    gdf_points,
    zcta[["ZCTA5CE20", "geometry"]],
    how="left",
    predicate="within"
)

data_2022_FINAL = gdf_with_zip.drop(
    columns=["geometry", "index_right"]
)

In [107]:
data_2022_FINAL

,Name,Building Number,Street,Unit,Fraction,Geo Area,City,State,Phone Number,Place Type,...,Geo Location,Lng,Lat,Website,Mapped Date,Updated Date,Is New Place,IsNormalized,tract_geoid,ZCTA5CE20
0,McDonald's,4556,N Kedzie Ave,NaN,NaN,Albany Park,Chicago,IL,773-583-4395,Dining,...,POINT(-87.7083965658057 41.9639705503226),-87.708397,41.963971,mcdonalds.com,7/25/2022 14:42,7/25/2022 14:42,NaN,False,17031140702,60625
1,Gt's Fast Food,5016,N Pulaski Rd,NaN,NaN,Albany Park,Chicago,IL,773-286-0050,Dining,...,POINT(-87.7285743 41.972321),-87.728574,41.972321,NaN,1/1/1900 0:00,1/1/1900 0:00,NaN,False,17031130200,60630
2,Jollibee,5033,N Elston Ave,NaN,NaN,Albany Park,Chicago,IL,773-685-6770,Dining,...,POINT(-87.7457287 41.9727333),-87.745729,41.972733,https://jollibeeusa.com,1/1/1900 0:00,1/1/1900 0:00,NaN,False,17031140400,60630
3,Grill City,5033,N Elston Ave,NaN,NaN,Albany Park,Chicago,IL,773-295-1658,Dining,...,POINT(-87.7457287 41.9727333),-87.745729,41.972733,https://grillcity.com,1/1/1900 0:00,1/1/1900 0:00,NaN,False,17031140400,60630
4,Sun Submarine,321,N Laramie Ave,NaN,NaN,Austin,Chicago,IL,773-378-8380,Dining,...,POINT(-87.7550463 41.886365),-87.755046,41.886365,NaN,1/1/1900 0:00,1/1/1900 0:00,NaN,False,17031251800,60644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7590,American Warriors,400,W 76th St,NaN,NaN,Greater Grand Crossing,Chicago,IL,NaN,Health Services,...,NaN,-87.635077,41.756471,http://www.drugfree.com/treatment-facility/347...,NaN,NaN,NaN,NaN,17031691200,60620
7591,Brite Kitchen,400,W 76th St,NaN,NaN,Greater Grand Crossing,Chicago,IL,NaN,Social Services & Political Advocacy,...,NaN,-87.635077,41.756471,NaN,NaN,NaN,NaN,NaN,17031691200,60620
7592,BZY Grill,130,E 79th St,NaN,NaN,Greater Grand Crossing,Chicago,IL,773-675-5551,Dining,...,NaN,-87.620088,41.751191,NaN,NaN,NaN,NaN,NaN,17031691300,60619
7593,Culver Transportation,400,W 76th St,NaN,NaN,Greater Grand Crossing,Chicago,IL,312-326-0511,"Wholesale, Storage and Transportation",...,NaN,-87.635077,41.756471,NaN,NaN,NaN,NaN,NaN,17031691200,60620


#### Aggregating the Datasets:

In [111]:
# 2020 Tract
data_2020_FINAL = data_2020_NEW
data_2020_active = data_2020_FINAL.loc[data_2020_FINAL['PlaceStatus'] == 'Active']

tracts_active_df_2020 = (
    data_2020_active
    .groupby(["tract_geoid", "PlaceType"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

tracts_active_df_2020 = tracts_active_df_2020.rename(columns = {'tract_geoid':'census_tract'})

tracts_active_df_2020 = tracts_active_df_2020.sort_values(["census_tract", "PlaceType"])

tracts_active_df_2020.to_csv('MAPSCorps_2020_Tract_PlaceType_Counts.csv',index = False)

In [113]:
# 2020 Zip

zip_active_df_2020 = (
    data_2020_active
    .groupby(["Zip", "PlaceType"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

zip_active_df_2020 = zip_active_df_2020.sort_values(["Zip", "PlaceType"])

zip_active_df_2020.to_csv('MAPSCorps_2020_Zip_PlaceType_Counts.csv',index = False)

In [117]:
#2021 Tract
data_2021_active = data_2021_FINAL.loc[data_2021_FINAL['Place Status'] == 'Active']

tracts_active_df_2021 = (
    data_2021_active
    .groupby(["tract_geoid", "Place Type"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

tracts_active_df_2021 = tracts_active_df_2021.rename(columns = {'tract_geoid':'census_tract'})

tracts_active_df_2021 = tracts_active_df_2021.sort_values(["census_tract", "Place Type"])

tracts_active_df_2021.to_csv('MAPSCorps_2021_Tract_PlaceType_Counts.csv',index = False)


In [122]:
# 2021 Zip

zip_active_df_2021 = (
    data_2021_active
    .groupby(["ZCTA5CE20", "Place Type"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

zip_active_df_2021 = zip_active_df_2021.rename(columns = {'ZCTA5CE20':'ZCTA_Zipcode'})

zip_active_df_2021 = zip_active_df_2021.sort_values(["ZCTA_Zipcode", "Place Type"])

zip_active_df_2021.to_csv('MAPSCorps_2021_Zip_PlaceType_Counts.csv',index = False)

In [123]:
#2022 Tract
data_2022_active = data_2022_FINAL.loc[data_2022_FINAL['Place Status'] == 'Active']

tracts_active_df_2022 = (
    data_2022_active
    .groupby(["tract_geoid", "Place Type"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

tracts_active_df_2022 = tracts_active_df_2022.rename(columns = {'tract_geoid':'census_tract'})

tracts_active_df_2022 = tracts_active_df_2022.sort_values(["census_tract", "Place Type"])

tracts_active_df_2022.to_csv('MAPSCorps_2022_Tract_PlaceType_Counts.csv',index = False)

In [124]:
# 2022 Zip

zip_active_df_2022 = (
    data_2022_active
    .groupby(["ZCTA5CE20", "Place Type"])
    .size()
    .reset_index(name="count_of_Active_PlaceStatus")
)

zip_active_df_2022 = zip_active_df_2022.rename(columns = {'ZCTA5CE20':'ZCTA_Zipcode'})

zip_active_df_2022 = zip_active_df_2022.sort_values(["ZCTA_Zipcode", "Place Type"])

zip_active_df_2022.to_csv('MAPSCorps_2022_Zip_PlaceType_Counts.csv',index = False)

In [125]:
data_2020_FINAL.to_csv('MAPSCorps_2020_Converted.csv',index = False)
data_2021_FINAL.to_csv('MAPSCorps_2021_Converted.csv',index = False)
data_2022_FINAL.to_csv('MAPSCorps_2022_Converted.csv',index = False)